<a href="https://colab.research.google.com/github/Adhiaris/Practical-Statistics-for-Data-Scientist-Books/blob/main/Practical_Statistics_Chapter7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Chapter 7: Unsupervised Learning

Unsupervised learning finds structure without labeled outcomes. Three main goals:
1. **Dimension reduction** — PCA
2. **Clustering** — K-Means, Hierarchical, GMM
3. **Exploratory insight** — understand variable relationships at scale

Data: S&P 500 stock returns and Lending Club loan records.


In [ ]:
!git clone -q --depth 1 https://github.com/gedeck/practical-statistics-for-data-scientists.git psds
!pip install -q prince adjustText

import math
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import cm
from matplotlib.colors import from_levels_and_colors
import seaborn as sns

from sklearn import preprocessing
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture

from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
from scipy.stats import multivariate_normal

import prince
from adjustText import adjust_text

import warnings
warnings.filterwarnings('ignore')

print("All imports successful.")

Key packages: `sklearn.decomposition.PCA`, `sklearn.cluster.KMeans`, `sklearn.mixture.GaussianMixture`, `scipy.cluster.hierarchy`, and `prince` for correspondence analysis.


## Load the Data

Three datasets:
1. `sp500_data.csv.gz` — daily % returns for hundreds of stocks.
2. `loan_data.csv.gz` — Lending Club borrower records.
3. `housetasks.csv` — contingency table of household chores.


In [ ]:
DATA = Path('psds/data')

sp500_px = pd.read_csv(DATA / 'sp500_data.csv.gz', index_col=0)
loan_data = pd.read_csv(DATA / 'loan_data.csv.gz')
housetasks = pd.read_csv(DATA / 'housetasks.csv', index_col=0)

print(f"S&P 500 data: {sp500_px.shape[0]} trading days x {sp500_px.shape[1]} stocks")
print(f"  Date range: {sp500_px.index[0]} to {sp500_px.index[-1]}")
print(f"Loan data: {loan_data.shape[0]} records x {loan_data.shape[1]} features")
print(f"House tasks: {housetasks.shape}")

Stock returns are already on a comparable scale, so standardization before PCA or K-Means is usually not needed here — revisited in the Scaling section.


---
# Principal Components Analysis (PCA)

PCA reduces $p$ correlated variables to uncorrelated **principal components**:
$$Z_i = w_{i,1}X_1 + w_{i,2}X_2 + \cdots + w_{i,p}X_p$$

$Z_1$ captures maximum variance; $Z_2$ captures the maximum remaining (orthogonal to $Z_1$), and so on. Works only with **numeric** variables.


## A Simple Example: Two Oil Stocks

With $p=2$ variables (XOM, CVX), there are exactly two principal components.


In [ ]:
oil_px = sp500_px[['XOM', 'CVX']]
print("Daily returns (first 5 rows):")
print(oil_px.head())

Both stocks often move together — their co-movement is what PC1 will capture.


In [ ]:
pcs = PCA(n_components=2)
pcs.fit(oil_px)
loadings = pd.DataFrame(pcs.components_, columns=oil_px.columns)
print("Component Loadings:")
print(loadings)
print(f"\nExplained variance ratio: {pcs.explained_variance_ratio_}")
print(f"Total variance explained by PC1: {pcs.explained_variance_ratio_[0]:.1%}")

**PC1** ($\approx$ equal weights, same sign) captures the **common energy-sector trend**. **PC2** (opposite signs) captures XOM vs. CVX **divergence**. PC1 explains the large majority of variance because both stocks are driven by crude oil prices.


In [ ]:
def abline(slope, intercept, ax):
    x_vals = np.array(ax.get_xlim())
    return (x_vals, intercept + slope * x_vals)

ax = oil_px.plot.scatter(x='XOM', y='CVX', alpha=0.3, figsize=(4, 4))
ax.set_xlim(-3, 3)
ax.set_ylim(-3, 3)
ax.plot(*abline(loadings.loc[0, 'CVX'] / loadings.loc[0, 'XOM'], 0, ax),
        '--', color='C1')
ax.plot(*abline(loadings.loc[1, 'CVX'] / loadings.loc[1, 'XOM'], 0, ax),
        '--', color='C1')
ax.set_title('PCA Directions: XOM vs CVX Returns')
plt.tight_layout()

**Figure:** Dashed lines show PCA directions. PC1 runs along the long axis (maximum spread); PC2 runs perpendicular. PCA rotates coordinates to align with maximum-variance directions.


### Connecting PCA to the Covariance Matrix

The eigenvalues of the covariance matrix equal the explained variances of the principal components.


In [ ]:
cov_matrix = oil_px.cov()
print("Covariance matrix:")
print(cov_matrix.round(4))

eigenvalues, eigenvectors = np.linalg.eigh(cov_matrix)
idx = eigenvalues.argsort()[::-1]
eigenvalues = eigenvalues[idx]

print(f"\nEigenvalues:        {eigenvalues}")
print(f"PCA explained_var_: {pcs.explained_variance_}")
print(f"Match: {np.allclose(eigenvalues, pcs.explained_variance_, atol=0.01)}")
print(f"\nTotal variance = sum(eigenvalues) = {eigenvalues.sum():.4f}")
print(f"  = Var(XOM) + Var(CVX) = {oil_px.var().sum():.4f}")

PCA is an eigendecomposition of the covariance matrix:
$$\text{Explained Variance Ratio}_i = \frac{\lambda_i}{\sum_j \lambda_j}$$

Eigenvalues sum to total variance — no information is lost, only rotated.


---
## 16-Stock Analysis

16 major stocks spanning technology, energy, finance, and consumer sectors.


In [ ]:
syms = sorted(['AAPL', 'MSFT', 'CSCO', 'INTC', 'CVX', 'XOM', 'SLB', 'COP',
        'JPM', 'WFC', 'USB', 'AXP', 'WMT', 'TGT', 'HD', 'COST'])
top_sp = sp500_px.loc[sp500_px.index >= '2011-01-01', syms]

sp_pca = PCA()
sp_pca.fit(top_sp)

print(f"Shape: {top_sp.shape} ({top_sp.shape[0]} days, {top_sp.shape[1]} stocks)")
print(f"Variance explained by first 5 PCs: {sp_pca.explained_variance_ratio_[:5].sum():.1%}")

cumvar = np.cumsum(sp_pca.explained_variance_ratio_)
for i in range(min(8, len(cumvar))):
    print(f"  PC1..{i+1}: {cumvar[i]:.1%}")

First 5 components explain a substantial fraction of total variance — significant dimensionality reduction from 16 to 5.


In [ ]:
explained_variance = pd.DataFrame(sp_pca.explained_variance_)
ax = explained_variance.head(10).plot.bar(legend=False, figsize=(4, 4))
ax.set_xlabel('Component')
ax.set_ylabel('Explained Variance (eigenvalue)')
ax.set_title('Screeplot: 16 S&P 500 Stocks')
plt.tight_layout()

**Screeplot:** The "elbow" where variance drops sharply then levels off indicates how many components to retain. A common rule: retain enough to explain 80% of variance.


In [ ]:
loadings = pd.DataFrame(sp_pca.components_[0:5, :], columns=top_sp.columns)

maxPC = 1.01 * loadings.loc[0:5, :].abs().to_numpy().max()
f, axes = plt.subplots(5, 1, figsize=(5, 5), sharex=True)
for i, ax in enumerate(axes):
    pc_loadings = loadings.loc[i, :]
    colors = ['C0' if l > 0 else 'C1' for l in pc_loadings]
    ax.axhline(color='#888888')
    pc_loadings.plot.bar(ax=ax, color=colors)
    ax.set_ylabel(f'PC{i+1}')
    ax.set_ylim(-maxPC, maxPC)
f.suptitle('Loadings: Top 5 Principal Components', y=1.02)
plt.tight_layout()

**PC1 ("Market"):** All same-sign loadings — captures overall market movement. **PC2 ("Energy vs. Rest"):** Energy stocks load opposite to others. **PC3–PC5:** Increasingly specific sector contrasts. PCA discovers market sector structure from covariance alone.


---
## Correspondence Analysis

For **categorical** data, **Correspondence Analysis** reveals associations in a contingency table — the categorical analogue of PCA.


In [ ]:
print("House tasks contingency table:")
print(housetasks)

In [ ]:
ca = prince.CA(n_components=2)
ca = ca.fit(housetasks)

ax = ca.row_coordinates(housetasks).plot.scatter(x=0, y=1, figsize=(6, 6))
ca.column_coordinates(housetasks).plot.scatter(x=0, y=1, ax=ax, c='C1')
texts = []
for idx, row in ca.row_coordinates(housetasks).iterrows():
    texts.append(plt.text(row[0], row[1], idx))
for idx, row in ca.column_coordinates(housetasks).iterrows():
    texts.append(plt.text(row[0], row[1], idx, color='C1'))
adjust_text(texts, only_move={'points':'y', 'texts':'y'})
ax.set_title('Correspondence Analysis: Household Tasks')
plt.tight_layout()

**CA biplot:** Horizontal axis separates wife-dominant from husband-dominant tasks; vertical separates solo from joint tasks. Tasks near a performer label are strongly associated with that person.


---
# K-Means Clustering

K-Means partitions $n$ records into $K$ clusters by minimizing **WCSS**:
$$\text{WCSS} = \sum_{k=1}^{K}\sum_{i\in C_k}\|\mathbf{x}_i - \boldsymbol{\mu}_k\|^2$$

**Algorithm:** Initialize $K$ centers → assign each point to nearest center → update centers to cluster means → repeat until convergence.


## Simple Example: XOM and CVX


In [ ]:
df = sp500_px.loc[sp500_px.index >= '2011-01-01', ['XOM', 'CVX']]
kmeans = KMeans(n_clusters=4, n_init='auto').fit(df)
df['cluster'] = kmeans.labels_
print("First 10 records with cluster assignments:")
print(df.head(10))

In [ ]:
centers = pd.DataFrame(kmeans.cluster_centers_, columns=['XOM', 'CVX'])
print("Cluster centers:")
print(centers)
print(f"\nCluster sizes: {dict(Counter(kmeans.labels_))}")

Two clusters have positive means (up-market days), two have negative means (down-market days). Magnitudes separate mild from extreme movements.


In [ ]:
fig, ax = plt.subplots(figsize=(4, 4))
ax = sns.scatterplot(x='XOM', y='CVX', hue='cluster', style='cluster',
                     ax=ax, data=df)
ax.set_xlim(-3, 3)
ax.set_ylim(-3, 3)
centers.plot.scatter(x='XOM', y='CVX', ax=ax, s=50, color='black')
ax.set_title('K-Means: 4 Clusters on XOM vs CVX')
plt.tight_layout()

**Figure:** K-Means produces Voronoi regions (convex, linear boundaries). Every record is assigned to a cluster — useful for regime-based partitioning even without well-separated groups.


## Interpreting Clusters: 16 Stocks with K=5


In [ ]:
syms = sorted(['AAPL', 'MSFT', 'CSCO', 'INTC', 'CVX', 'XOM', 'SLB', 'COP',
               'JPM', 'WFC', 'USB', 'AXP', 'WMT', 'TGT', 'HD', 'COST'])
top_sp = sp500_px.loc[sp500_px.index >= '2011-01-01', syms]
kmeans = KMeans(n_clusters=5, n_init='auto').fit(top_sp)
print(f"Cluster sizes: {Counter(kmeans.labels_)}")

In [ ]:
centers = pd.DataFrame(kmeans.cluster_centers_, columns=syms)
f, axes = plt.subplots(5, 1, figsize=(5, 6), sharex=True)
for i, ax in enumerate(axes):
    center = centers.loc[i, :]
    maxPC = 1.01 * np.max(np.max(np.abs(center)))
    colors = ['C0' if l > 0 else 'C1' for l in center]
    ax.axhline(color='#888888')
    center.plot.bar(ax=ax, color=colors)
    ax.set_ylabel(f'Cluster {i + 1}')
    ax.set_ylim(-maxPC, maxPC)
f.suptitle('K-Means Cluster Centers (K=5)', y=1.02)
plt.tight_layout()

**Cluster centers:** Each subplot shows mean return per stock within a cluster. Positive = stocks up; negative = stocks down. Some clusters capture broad market moves; others capture sector rotations.


## Selecting K: The Elbow Method

Plot average within-cluster distance vs. $K$. The "elbow" — where adding clusters yields diminishing returns — guides the choice.


In [ ]:
inertia = []
for n_clusters in range(2, 15):
    kmeans = KMeans(n_clusters=n_clusters, random_state=0, n_init='auto').fit(top_sp)
    inertia.append(kmeans.inertia_ / n_clusters)
inertias = pd.DataFrame({'n_clusters': range(2, 15), 'inertia': inertia})
ax = inertias.plot(x='n_clusters', y='inertia')
plt.xlabel('Number of clusters (k)')
plt.ylabel('Average Within-Cluster Squared Distances')
plt.ylim((0, 1.1 * inertias.inertia.max()))
ax.legend().set_visible(False)
plt.title('Elbow Method for K-Means')
plt.tight_layout()

No clear elbow here — typical for noisy data. In practice, $K$ is often driven by **business requirements** (e.g., how many customer segments to manage) rather than a statistical criterion.


---
# Hierarchical Clustering

Builds a **dendrogram** without pre-specifying $K$.

**Agglomerative (bottom-up):** Start with each record as its own cluster → merge the two most similar → repeat.

Linkage types:
$$D_{\text{complete}} = \max d(a_i,b_j), \quad D_{\text{single}} = \min d(a_i,b_j), \quad D_{\text{avg}} = \text{mean of all } d(a_i,b_j)$$


## Clustering 18 Companies


In [ ]:
syms1 = ['AAPL', 'AMZN', 'AXP', 'COP', 'COST', 'CSCO', 'CVX', 'GOOGL', 'HD',
         'INTC', 'JPM', 'MSFT', 'SLB', 'TGT', 'USB', 'WFC', 'WMT', 'XOM']
df = sp500_px.loc[sp500_px.index >= '2011-01-01', syms1].transpose()
print(f"Shape: {df.shape} ({df.shape[0]} companies x {df.shape[1]} trading days)")

Z = linkage(df, method='complete')
print(f"Linkage matrix: {Z.shape} ({Z.shape[0]} merges)")

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
dendrogram(Z, labels=list(df.index), color_threshold=0)
plt.xticks(rotation=90)
ax.set_ylabel('Distance')
ax.set_title('Dendrogram: 18 Companies (Complete Linkage)')
plt.tight_layout()

**Dendrogram:** Oil stocks (SLB, CVX, XOM, COP) merge early — very similar. GOOGL and AMZN stand apart. To extract clusters, cut the tree at a chosen height.


In [ ]:
memb = fcluster(Z, 4, criterion='maxclust')
memb = pd.Series(memb, index=df.index)
print("Cluster memberships (4 clusters):")
for key, item in memb.groupby(memb):
    print(f"  Cluster {key}: {', '.join(item.index)}")

Hierarchy reveals structure at multiple granularities — a key advantage over K-Means.


## Comparing Linkage Methods


In [ ]:
df = sp500_px.loc[sp500_px.index >= '2011-01-01', ['XOM', 'CVX']]
fig, axes = plt.subplots(nrows=2, ncols=2, figsize=(5, 5))
for i, method in enumerate(['single', 'average', 'complete', 'ward']):
    ax = axes[i // 2, i % 2]
    Z = linkage(df, method=method)
    colors = [f'C{c+1}' for c in fcluster(Z, 4, criterion='maxclust')]
    ax = sns.scatterplot(x='XOM', y='CVX', hue=colors, style=colors,
                         size=0.5, ax=ax, data=df, legend=False)
    ax.set_xlim(-3, 3)
    ax.set_ylim(-3, 3)
    ax.set_title(method)
plt.suptitle('4 Linkage Methods (4 Clusters Each)', y=1.02)
plt.tight_layout()

**Four linkage methods produce very different clusters:**
- **Single:** Chaining effect — one big cluster plus outlier singletons.
- **Average:** Compromise between single and complete.
- **Complete:** Requires all pairs to be close — compact clusters.
- **Ward's:** Minimizes within-cluster variance — most similar to K-Means.

Use **complete** or **Ward's** for most applications.


---
# Model-Based Clustering (Gaussian Mixture Models)

Assumes data was generated by $K$ multivariate normals:
$$p(\mathbf{x}) = \sum_{k=1}^{K} \pi_k \cdot \mathcal{N}(\mathbf{x}\mid\boldsymbol{\mu}_k, \boldsymbol{\Sigma}_k)$$

Parameters estimated via **Expectation-Maximization (EM)**.


In [ ]:
mean = [0.5, -0.5]
cov = [[1, 1], [1, 2]]
probability = [.5, .75, .95, .99]

def probLevel(p):
    D = 1
    return (1 - p) / (2 * math.pi * D)
levels = [probLevel(p) for p in probability]

fig, ax = plt.subplots(figsize=(5, 5))
x, y = np.mgrid[-2.8:3.8:.01, -5:4:.01]
pos = np.empty(x.shape + (2,))
pos[:, :, 0] = x; pos[:, :, 1] = y
rv = multivariate_normal(mean, cov)

CS = ax.contourf(x, y, rv.pdf(pos), cmap=cm.GnBu, levels=50)
ax.contour(CS, levels=levels, colors=['black'])
ax.plot(*mean, color='black', marker='o')
ax.set_xlabel('X')
ax.set_ylabel('Y')
ax.set_title('Multivariate Normal: μ=(0.5, -0.5)')
plt.tight_layout()

**Figure:** Probability contours for a 2D normal. The elliptical shape reflects positive covariance ($\sigma_{xy}=1$). Contour lines show 50%, 75%, 95%, and 99% probability regions.


## Gaussian Mixture on Stock Returns


In [ ]:
df = sp500_px.loc[sp500_px.index >= '2011-01-01', ['XOM', 'CVX']]
mclust = GaussianMixture(n_components=2).fit(df)
print(f"BIC: {mclust.bic(df):.2f}")

fig, ax = plt.subplots(figsize=(4, 4))
colors = [f'C{c}' for c in mclust.predict(df)]
df.plot.scatter(x='XOM', y='CVX', c=colors, alpha=0.5, ax=ax)
ax.set_xlim(-3, 3)
ax.set_ylim(-3, 3)
ax.set_title('Gaussian Mixture: 2 Components')
plt.tight_layout()

**Figure:** GMM finds a low-volatility core and a high-volatility outer group — unlike K-Means (spatial quadrants), GMM models the **shape** (covariance) of each cluster.


In [ ]:
print('Means:')
print(mclust.means_)
print('\nCovariances:')
print(mclust.covariances_)
print(f'\nMixing weights: {mclust.weights_}')

The two components have similar means (near zero) but very different covariances. The wide-variance component captures extreme/volatile days — reflecting stock returns' **heavy tails**.


## Selecting Components with BIC

$$\text{BIC} = -2\ln(\hat{L}) + k\ln(n)$$

Lower BIC = better balance of fit and parsimony. Test across covariance types and component counts.


In [ ]:
results = []
covariance_types = ['full', 'tied', 'diag', 'spherical']
for n_components in range(1, 9):
    for covariance_type in covariance_types:
        mclust = GaussianMixture(n_components=n_components, warm_start=True,
                                 covariance_type=covariance_type)
        mclust.fit(df)
        results.append({
            'bic': mclust.bic(df),
            'n_components': n_components,
            'covariance_type': covariance_type,
        })

results = pd.DataFrame(results)
styles = ['C0-','C1:','C0-.', 'C1--']

fig, ax = plt.subplots(figsize=(4, 4))
for i, ct in enumerate(covariance_types):
    subset = results.loc[results.covariance_type == ct, :]
    subset.plot(x='n_components', y='bic', ax=ax, label=ct, kind='line', style=styles[i])
ax.set_xlabel('Number of Components')
ax.set_ylabel('BIC')
ax.set_title('BIC: Gaussian Mixture Model Selection')
plt.tight_layout()

best = results.loc[results.bic.idxmin()]
print(f"Best: {best['covariance_type']} with {int(best['n_components'])} components (BIC={best['bic']:.1f})")

**Figure:** BIC can show a clear minimum — unlike the K-Means elbow — giving a more principled answer to "how many clusters?"


---
# Scaling and Categorical Variables

All unsupervised methods are sensitive to variable **scale**. **Standardize** unless variables are already comparable:
$$z = \frac{x - \bar{x}}{s}$$


## Without Scaling vs. With Scaling


In [ ]:
loan_data['outcome'] = pd.Categorical(loan_data['outcome'],
                                      categories=['paid off', 'default'], ordered=True)
defaults = loan_data.loc[loan_data['outcome'] == 'default',]
columns = ['loan_amnt', 'annual_inc', 'revol_bal', 'open_acc', 'dti', 'revol_util']

df = defaults[columns]
kmeans = KMeans(n_clusters=4, random_state=1, n_init='auto').fit(df)
counts = Counter(kmeans.labels_)
centers = pd.DataFrame(kmeans.cluster_centers_, columns=columns)
centers['size'] = [counts[i] for i in range(4)]
print("UNSCALED cluster centers:")
print(centers.round(1))

Without scaling, `annual_inc` and `revol_bal` dominate. One cluster has very few members — high-income outliers isolated by numeric magnitude, not meaningful borrower differences.


In [ ]:
scaler = preprocessing.StandardScaler()
df0 = scaler.fit_transform(df * 1.0)
kmeans = KMeans(n_clusters=4, random_state=1, n_init='auto').fit(df0)
counts = Counter(kmeans.labels_)
centers = pd.DataFrame(scaler.inverse_transform(kmeans.cluster_centers_), columns=columns)
centers['size'] = [counts[i] for i in range(4)]
print("SCALED cluster centers (inverse-transformed):")
print(centers.round(1))

After scaling, clusters are balanced across all variables and tell a meaningful story: high-DTI vs. low-DTI, high utilization vs. low. **Always standardize before PCA, K-Means, or hierarchical clustering.**


## Dominant Variables in PCA

Even on the same unit (stock returns), high-variance stocks can dominate PCA.


In [ ]:
syms = ['GOOGL', 'AMZN', 'AAPL', 'MSFT', 'CSCO', 'INTC', 'CVX', 'XOM',
        'SLB', 'COP', 'JPM', 'WFC', 'USB', 'AXP', 'WMT', 'TGT', 'HD', 'COST']
top_sp1 = sp500_px.loc[sp500_px.index >= '2005-01-01', syms]

sp_pca1 = PCA()
sp_pca1.fit(top_sp1)

explained_variance = pd.DataFrame(sp_pca1.explained_variance_)
ax = explained_variance.head(10).plot.bar(legend=False, figsize=(4, 4))
ax.set_xlabel('Component')
ax.set_ylabel('Explained Variance')
ax.set_title('Screeplot: Including GOOGL and AMZN')
plt.tight_layout()

In [ ]:
loadings = pd.DataFrame(sp_pca1.components_[0:2, :], columns=top_sp1.columns)
print("Top 2 PC loadings:")
print(loadings.transpose().round(3))

GOOGL and AMZN swamp the first two PCs. Solutions: standardize, exclude dominant variables, or analyze them separately.


## Problems with Clustering Mixed Data


In [ ]:
columns = ['dti', 'payment_inc_ratio', 'home_', 'pub_rec_zero']
df = pd.get_dummies(defaults[columns], dtype=int)

scaler = preprocessing.StandardScaler()
df0 = scaler.fit_transform(df * 1.0)
kmeans = KMeans(n_clusters=4, random_state=1, n_init='auto').fit(df0)
centers = pd.DataFrame(scaler.inverse_transform(kmeans.cluster_centers_), columns=df.columns)
print("Cluster centers (mixed data, one-hot encoded):")
print(centers.round(2))